Before EGU 2026, I decided to re-make the passage of OPT operator derivation since we need to take care of staggered grids etc. which are not easy to handle with the current version

# Big change!!

pointsInSpace, pointsInTime should be real (before they were -1 but now it should be 2,3,4 ...)

In [1]:
using Pkg
using Metal

cd(@__DIR__)
Pkg.activate("../")
ParamFile = "../config/testparam.csv"  # maybe GeoPoints and planet1D should be fusioned

# batchGPU should be at this level (I have not made it as a module yet, since the choice of Metal/CUDA should be done in a manual way)
include("../src/batchFiles/batchGPU.jl")


include("../src/commonBatchs.jl")

include("../src/planet1D.jl")
include("../src/GeoPoints.jl")

using .commonBatchs, .planet1D, .GeoPoints

  Activating project at `~/Documents/Github/flexOPT`


devs = Metal.devices() = Metal.MTL.MTLDeviceInstance[Metal.MTL.MTLDeviceInstance (object of type AGXG13XDevice)]
→ Using Metal backend (1 device(s))
Selected backend type: MetalBackend


In [2]:
# Since this is the developing version, I do not use the module from flexOPT.jl

use some solid packages

In [3]:
include("../src/motorsOPT/others.jl")

BouncingCoordinates (generic function with 1 method)

In [4]:
    include("../src/motorsOPT/famousEquations.jl")

famousEquation (generic function with 15 methods)

input parameters

In [ ]:
famousEquationType="2DacousticTime"
Δ = (1.0,1.0,1.0)

# orders: -1 -> indicator function, 0 -> box car, >=1 -> B-spline

orderBtime=-1
orderBspace=-1
pointsInSpace=3
pointsInTime=3

# new parameters for interpolated Taylor expansion
# μ points should be distributed from y_min+offsetμFromExtremitiesInΔy*Δy to y_max-offsetμFromExtremitiesInΔy*Δy
pointsμInSpace = 2
pointsμInTime = 2
offsetμFromExtremitiesInΔyInSpace=1//2
offsetμFromExtremitiesInΔyInTime=1//2

YorderBspace=-1
YorderBtime=-1
supplementaryOrder=2

2

We also propose different ways of boundary treatment (ghosting: Neumann/Dirichlet; same number of points etc.)

future functions

Starting the OPT derivation

In [ ]:

Δnum = SVector(Δ)

concreteParametersForOPTConstruction = @strdict famousEquationType Δnum orderBtime orderBspace YorderBtime YorderBspace supplementaryOrder pointsInSpace pointsInTime 

# construction of NamedTuples
trialFunctionsCharacteristics=(orderBtime=orderBtime,orderBspace=orderBspace,pointsInSpace=pointsInSpace,pointsInTime=pointsInTime)
TaylorOptions=(YorderBtime=YorderBtime,YorderBspace=YorderBspace,supplementaryOrder=supplementaryOrder,pointsμInSpace=pointsμInSpace,pointsμInTime=pointsμInTime,offsetμInΔyInSpace=offsetμFromExtremitiesInΔyInSpace,offsetμInΔyInTime=offsetμFromExtremitiesInΔyInTime)


(WorderBtime = -1, WorderBspace = -1, supplementaryOrder = 2, pointsμInSpace = 2, pointsμInTime = 2, offsetμInΔyInSpace = 1//2, offsetμInΔyInTime = 1//2)

In [14]:
equationCharacteristics=famousEquations(famousEquationType)
numbersOfTheSystem=numbersOfTheExpression(equationCharacteristics,trialFunctionsCharacteristics,TaylorOptions)
dependencies,ordersForSplines,configsTaylor=investigateDependencies(equationCharacteristics,numbersOfTheSystem,trialFunctionsCharacteristics,TaylorOptions)
bigα,varM=bigαFinder(equationCharacteristics,numbersOfTheSystem,ordersForSplines)
@show bigα,varM
@show configsTaylor


(bigα, varM) = (Any[Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))];;], Any[v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉ v₁ v₂ v₃ v₄ v₅ v₆ v₇ v₈ v₉])
configsTaylor = (numberGeometries = 1, multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = Any[SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14], availableμPoints = Any[SVector{3, Float64}[[0.5, 0.5, 0.5] [0.5, 1.5, 0.5]; [1.5, 0.5, 0.5] [1.5, 1.5, 0.5];;; [0.5, 0.5, 1.5] [0.5, 1.5, 1.5]; [1.5, 0.5, 1.5] [1.5, 1.5, 1.5]]])


(numberGeometries = 1, multiOrdersIndices = CartesianIndices((5, 5, 5)), availablePointsConfigurations = Any[SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]], centrePointConfigurations = [14], availableμPoints = Any[SVector{3, Float64}[[0.5, 0.5, 0.5] [0.5, 1.5, 0.5]; [1.5, 0.5, 0.5] [1.5, 1.5, 0.5];;; [0.5, 0.5, 1.5] [0.5, 1.5, 1.5]; [1.5, 0.5, 1.5] [1.5, 1.5, 1.5]]])

In [32]:
for iConfigGeometry in 1:configsTaylor.numberGeometries
       @show pointsIndices=configsTaylor.availablePointsConfigurations[iConfigGeometry]
    @show middleLinearν=configsTaylor.centrePointConfigurations[iConfigGeometry]
    @show typeof(pointsIndices[middleLinearν])
   
end

pointsIndices = configsTaylor.availablePointsConfigurations[iConfigGeometry] = SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]
middleLinearν = configsTaylor.centrePointConfigurations[iConfigGeometry] = 14
typeof(pointsIndices[middleLinearν]) = SVector{3, Int64}


In [ ]:
function constructAmatrix_develop(iConfigGeometry,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor)
    
    
    #(coordinates,multiOrdersIndices,pointsIndices,multiPointsIndices,middleLinearν,Δ,varM,bigα,orderBspline,YorderBspline,NtypeofExpr,NtypeofFields;threads = 256,backend=backend)

    # This function is for one iConfigGeometry

    #region preparation

    @unpack nCoordinates = numbersOfTheSystem

    @unpack multiOrdersIndices,availablePointsConfigurations,centrePointConfigurations,availableμPoints = configsTaylor

  

    @show pointsIndices=availablePointsConfigurations[iConfigGeometry] # CartesianIndices
    @show middleLinearν=centrePointConfigurations[iConfigGeometry] # scalar
    @show μPoints = availableμPoints[iConfigGeometry] # Array(SVector)
    @show pointν = pointsIndices[middleLinearν] # SVector
    

    # this is fully GPU optimised version of ASymbolic 
    
    nPoints = length(pointsIndices)
    nLs = length(multiOrdersIndices)

    # 

    L_MINUS_N = multiOrdersIndices
    L_MINUS_N = L_MINUS_N .-L_MINUS_N[1] 

    # here L_MINUS_N is truly \mathbf{l}-\mahtbf{n} ∈ \mathbb{Z}_{≥0}

    #endregion

    #region we compute the integral of WYYKK in 1D domain (μ ∈ \mathbb{Z}/2 has not yet been implemented)

    
    # look at the debug1DKernelIntegral.ipynb    



    #integral1DWYYKK = Array{Any,1}(undef,nCoordinates)
    #modifiedμ=Array{Any,1}(undef,nCoordinates)
    #for iCoord in eachindex(coordinates) # for each 
    #    integralParams = @strdict oB =orderBspline[iCoord] oWB = YorderBspline[iCoord] νCoord=pointsIndices[middleLinearν][iCoord] LCoord = multiPointsIndices[end][iCoord] ΔCoord=Δ[iCoord] l_n_max=L_MINUS_N[end][iCoord]
    #    output = myProduceOrLoad(getIntegralWYYKK,integralParams,"intKernel")
    #    integral1DWYYKK[iCoord] = output["intKernelforνLΔ"]
    #    modifiedμ[iCoord] = output["modμ"] # this can be still 'nothing'
    #end


end
    #endregion

constructAmatrix_develop (generic function with 1 method)

In [38]:
constructAmatrix_develop(1,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor)
    

pointsIndices = availablePointsConfigurations[iConfigGeometry] = SVector{3, Int64}[[1, 1, 1] [1, 2, 1] [1, 3, 1]; [2, 1, 1] [2, 2, 1] [2, 3, 1]; [3, 1, 1] [3, 2, 1] [3, 3, 1];;; [1, 1, 2] [1, 2, 2] [1, 3, 2]; [2, 1, 2] [2, 2, 2] [2, 3, 2]; [3, 1, 2] [3, 2, 2] [3, 3, 2];;; [1, 1, 3] [1, 2, 3] [1, 3, 3]; [2, 1, 3] [2, 2, 3] [2, 3, 3]; [3, 1, 3] [3, 2, 3] [3, 3, 3]]
middleLinearν = centrePointConfigurations[iConfigGeometry] = 14
μPoints = availableμPoints[iConfigGeometry] = SVector{3, Float64}[[0.5, 0.5, 0.5] [0.5, 1.5, 0.5]; [1.5, 0.5, 0.5] [1.5, 1.5, 0.5];;; [0.5, 0.5, 1.5] [0.5, 1.5, 1.5]; [1.5, 0.5, 1.5] [1.5, 1.5, 1.5]]
pointν = pointsIndices[middleLinearν] = [2, 2, 2]


CartesianIndices((0:4, 0:4, 0:4))

In [39]:
ordersForSplines

(orderBspline = [-1, -1, -1], WorderBspline = [-1, -1, -1], orderExpressions = [3, 3, 3], orderU = [5, 5, 5])

In [ ]:
function AmatrixSemiSymbolicGPU(iConfigGeometry,numbersOfTheSystem,dependencies,ordersForSplines,configsTaylor)
    
    
    #(coordinates,multiOrdersIndices,pointsIndices,multiPointsIndices,middleLinearν,Δ,varM,bigα,orderBspline,YorderBspline,NtypeofExpr,NtypeofFields;threads = 256,backend=backend)

    # This function is for one iConfigGeometry

    #region preparation

    @unpack nCoordinates = numbersOfTheSystem

    @unpack multiOrdersIndices,availablePointsConfigurations,availableμPoints = configsTaylor

  

    @show pointsIndices=availablePointsConfigurations[iConfigGeometry] # CartesianIndices
    @show middleLinearν=centrePointConfigurations[iConfigGeometry] # scalar
    @show μPoints = availableμPoints[iConfigGeometry] # Array(SVector)
    @show pointν = pointsIndices[middleLinearν] # SVector
    

    # this is fully GPU optimised version of ASymbolic 
    
    nPoints = length(pointsIndices)
    nLs = length(multiOrdersIndices)

    # 

    L_MINUS_N = multiOrdersIndices
    L_MINUS_N = L_MINUS_N .-L_MINUS_N[1]
    # here L_MINUS_N is truly \mathbf{l}-\mahtbf{n} ∈ \mathbb{Z}_{≥0}

    #endregion

    #region we compute the integral of WYYKK in 1D domain (μ ∈ \mathbb{Z}/2 has not yet been implemented)

    



    integral1DWYYKK = Array{Any,1}(undef,nCoordinates)
    modifiedμ=Array{Any,1}(undef,nCoordinates)
    for iCoord in eachindex(coordinates) # for each 
        integralParams = @strdict oB =orderBspline[iCoord] oWB = YorderBspline[iCoord] νCoord=pointsIndices[middleLinearν][iCoord] LCoord = multiPointsIndices[end][iCoord] ΔCoord=Δ[iCoord] l_n_max=L_MINUS_N[end][iCoord]
        output = myProduceOrLoad(getIntegralWYYKK,integralParams,"intKernel")
        integral1DWYYKK[iCoord] = output["intKernelforνLΔ"]
        modifiedμ[iCoord] = output["modμ"] # this can be still 'nothing'
    end



    #endregion

    #region get CˡηGlobal (for ν)

    coefInversionDict = @strdict coordinates multiOrdersIndices pointsIndices Δ YorderBspline modifiedμ

    #@show coordinates multiOrdersIndices pointsIndices Δ YorderBspline modifiedμ

    output=myProduceOrLoad(TaylorCoefInversion,coefInversionDict,"taylorCoefInv")
    Cˡη=output["CˡηGlobal"]

    #endregion 

    #region objectives (Ajiννᶜ)

    # the order is: (νᶜ,) ν, i, j  here
    Ajiννᶜ = Array{Num,4}(undef,nPoints,nPoints,NtypeofFields,NtypeofExpr)

    # but this will already include bigα so the coefficients for each α_{nn'ji} should be given here

    #endregion

    #region useful LinearIndices conversion functions
    
    LI_points = LinearIndices(pointsIndices)
    LI_L_MINUS_N_plus_1 = LinearIndices(L_MINUS_N.+vec2car(ones(Int,nCoordinates)))
  
    #endregion

    #region make the table for each (x',x,n',n) (x=η+μ)
    nTotalSmallα = sum(length(bigα[iExpr, iField]) for iExpr in 1:NtypeofExpr, iField in 1:NtypeofFields)

    tableForLoop = Array{Int32,3}(undef,2+nCoordinates*2,nLs*nLs,nTotalSmallα) # for every α it will give the l and l-n
    
    fill!(tableForLoop, 0)
    indexLinearα = 1
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        
        α = bigα[iExpr,iField]
        for eachα ∈ α
            nᶜ = eachα.nᶜ - vec2car(ones(Int,nCoordinates))
            n = eachα.n - vec2car(ones(Int,nCoordinates))
            #nodeValue = eachα.node # not important at this point
            # Available indices
            Lᶜ_avail = (nᶜ .+ L_MINUS_N) ∩ L_MINUS_N
            L_avail = (n .+ L_MINUS_N) ∩ L_MINUS_N
            Lᶜ_Nᶜ_avail = Lᶜ_avail .- nᶜ
            L_N_avail = L_avail .- n
            
            iL=1
            for lᶜ ∈ Lᶜ_avail, l ∈ L_avail
                
                tableForLoop[1,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ+vec2car(ones(Int,nCoordinates))]
                tableForLoop[2,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[3,iL,indexLinearα]= LI_L_MINUS_N_plus_1[lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))]
                #tableForLoop[4,iL,indexLinearα]= LI_L_MINUS_N_plus_1[l-n+vec2car(ones(Int,nCoordinates))]
                tmplᶜ_nᶜ = lᶜ-nᶜ+vec2car(ones(Int,nCoordinates))
                tmpl_n = l-n+vec2car(ones(Int,nCoordinates))
                for iCoord ∈ 1:nCoordinates
                    tableForLoop[2+iCoord,iL,indexLinearα] = tmplᶜ_nᶜ[iCoord]
                    tableForLoop[2+iCoord+nCoordinates,iL,indexLinearα] = tmpl_n[iCoord]
                end

                iL += 1
            end
       
            indexLinearα += 1
        end
    end


    #endregion

    #region make a dictionary for μ ∈ pointsIndices and its linearised version

    tableForPoints = Array{Int32,2}(undef,nCoordinates,nPoints)

    for μ ∈ pointsIndices, iCoord ∈ 1:nCoordinates
        linearisedμ = LI_points[vec2car(μ)]
        tableForPoints[iCoord,linearisedμ]=vec2car(μ)[iCoord]
    end

    #endregion

    #region adapt the arrays to the GPU backend
    tableForPoints_gpu = Adapt.adapt(backend,tableForPoints)
    tableForLoop_gpu = Adapt.adapt(backend,tableForLoop)
    C_gpu = Adapt.adapt(backend,Float32.(Cˡη))


    # Collect the size of each array
    all_sizes = collect.(size.(integral1DWYYKK))  # vector of tuples

    # Element-wise maximum across all dimensions
    max_size = map((xs...) -> maximum(xs), all_sizes...)

    int_total_float32 = Array{Float32,5}(undef, nCoordinates, max_size...)
    int_total_float32 .= 0.f0
    for iCoord in 1:nCoordinates
        small_size = size(integral1DWYYKK[iCoord])
        tmpMatrix = Float32.(integral1DWYYKK[iCoord])
        tmpRange = CartesianIndices(tmpMatrix)
        int_total_float32[iCoord,tmpRange] = tmpMatrix
    end
    int_gpu = Adapt.adapt(backend,int_total_float32)
    


    output_gpu = Adapt.adapt(backend, zeros(Float32, nPoints, nPoints, nTotalSmallα))
    
    #


    #endregion
    
    #region launch GPU computation

    # scalars in Int32
    P      = Int32(nPoints)
    L      = Int32(nLs)
    nDim   = Int32(nCoordinates)
    nα     = Int32(nTotalSmallα)
    int1max = Int32(max_size[1])
    int2max = Int32(max_size[2])

    #@show typeof(output_gpu)         # must be MtlArray{Float32,3}
    #@show typeof(C_gpu)              # must be MtlArray{Float32,3}
    #@show typeof(int_gpu)            # must be MtlArray{Float32,5}
    #@show typeof(tableForLoop_gpu)   # must be MtlArray{Int32,3}
    #@show typeof(tableForPoints_gpu) # must be MtlMatrix{Int32}
    #@show typeof(P) typeof(L) typeof(nDim) typeof(nα) typeof(int1max) typeof(int2max)

    kernel! = windowContraction!(backend,(8,8,8))#,128,size(output_gpu))
    kernel!(output_gpu, C_gpu, int_gpu, tableForLoop_gpu,tableForPoints_gpu,
       P,L,nDim,nα,int1max,int2max,ndrange=size(output_gpu))
    KernelAbstractions.synchronize(backend)

    # Here output_gpu[x',x,eachα] ∑μ' ∑μ ∑l' ∑ l C[x',l',μ'] C[x,l,μ] ∏_iCoord K[iCoord][μ',μ,l'-n',l-n]
    # with (n',n) depends on eachα and x=η+μ

    @show "GPU computation of Ajiννᶜ: done"
    output = Adapt.adapt(CPU(), output_gpu)
    #endregion


    #region contruct Ulocal

    # the order is: (νᶜ,) ν, i, j  here

    Ulocal = Array{Num,2}(undef,length(pointsIndices),NtypeofFields)
    for iField in eachindex(fields)
        newstring=split(string(fields[iField]),"(")[1]
        Ulocal[:,iField]=Symbolics.variables(Symbol(newstring),1:length(pointsIndices))
    end

    #endregion

    #region make Ajiννᶜ and AjiννᶜU symbolically (which we will soon remove!)
    Ajiννᶜ = Array{Num,3}(undef,length(pointsIndices),NtypeofFields,NtypeofExpr)
    AjiννᶜU = Array{Num,1}(undef,NtypeofExpr)
    
    # this is the cost function for ν point so the number of elements is just the number of expressions (governing equations)
    Ajiννᶜ .= 0
    AjiννᶜU .= 0
    indexLinearα = 1
   
    for iExpr ∈ 1:NtypeofExpr,iField ∈ 1:NtypeofFields
        α = bigα[iExpr,iField]
        for eachα ∈ α
            @show nodeValue=eachα.node
            for x ∈ pointsIndices
                xLinear = LI_points[vec2car(x)]
                
                localmapηᶜ=Dict()
                for iVar ∈ eachindex(vars)
                    localmapηᶜ[vars[iVar]]=varM[iVar,xLinear][]
                end
                for xᶜ ∈ pointsIndices
                    xᶜLinear = LI_points[vec2car(xᶜ)]
                    U_HERE = Ulocal[xᶜLinear,iField]                    
                    substitutedValue = substitute(nodeValue, localmapηᶜ)
                    Ajiννᶜ[xᶜLinear,iField,iExpr] += Float64(output[xᶜLinear,xLinear,indexLinearα])*substitutedValue
                    AjiννᶜU[iExpr]+= Ajiννᶜ[xᶜLinear,iField,iExpr] * U_HERE
                end

            end
            indexLinearα += 1
        end

    end


    #endregion


    return AjiννᶜU,Ulocal, output
end 

AmatrixSemiSymbolicGPU (generic function with 2 methods)

In [20]:
bigα

1×1 Matrix{Any}:
 Any[(node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(3, 1, 1)), (node = -(v(x, y)^2), nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 3, 1)), (node = 1, nᶜ = CartesianIndex(1, 1, 1), n = CartesianIndex(1, 1, 3))]

In [21]:
configsTaylor.availablePointsConfigurations[1]

3×3×3 Array{SVector{3, Int64}, 3}:
[:, :, 1] =
 [1, 1, 1]  [1, 2, 1]  [1, 3, 1]
 [2, 1, 1]  [2, 2, 1]  [2, 3, 1]
 [3, 1, 1]  [3, 2, 1]  [3, 3, 1]

[:, :, 2] =
 [1, 1, 2]  [1, 2, 2]  [1, 3, 2]
 [2, 1, 2]  [2, 2, 2]  [2, 3, 2]
 [3, 1, 2]  [3, 2, 2]  [3, 3, 2]

[:, :, 3] =
 [1, 1, 3]  [1, 2, 3]  [1, 3, 3]
 [2, 1, 3]  [2, 2, 3]  [2, 3, 3]
 [3, 1, 3]  [3, 2, 3]  [3, 3, 3]

In [22]:
configsTaylor.availableμPoints[1]

2×2×2 Array{SVector{3, Float64}, 3}:
[:, :, 1] =
 [0.5, 0.5, 0.5]  [0.5, 1.5, 0.5]
 [1.5, 0.5, 0.5]  [1.5, 1.5, 0.5]

[:, :, 2] =
 [0.5, 0.5, 1.5]  [0.5, 1.5, 1.5]
 [1.5, 0.5, 1.5]  [1.5, 1.5, 1.5]

In [23]:
typeof(configsTaylor.availablePointsConfigurations[1])

Array{SVector{3, Int64}, 3}

In [24]:
configsTaylor.availablePointsConfigurations[1][configsTaylor.centrePointConfigurations[1]]

3-element SVector{3, Int64} with indices SOneTo(3):
 2
 2
 2

In [25]:
@show numbersOfTheSystem.pointsμUsed
@show numbersOfTheSystem.offsetsμUsed

numbersOfTheSystem.pointsμUsed = [2, 2, 2]
numbersOfTheSystem.offsetsμUsed = [0.5, 0.5, 0.5]


3-element Vector{Float64}:
 0.5
 0.5
 0.5